In [1]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [ ]:
from pathlib import Path

from IPython.display import display
import polars as pl
import warnings

from torch.utils.data import DataLoader

from tabseq_torch.dl import (
    DDChunkReader,
    DataCatalog,
    LazyDataset,
    cat_embedding_params,
    collate_padded_batch_fn,
    random_split,
    read_json,
    read_num_grous,
)
from tabseq_torch.lit import LitImbBinTabLSTM, LogPRMethod, PoolingType


In [3]:
tx_path = Path(
    "/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/train_transactions_contest"
)
print(tx_path.exists())
for p in tx_path.parent.iterdir():
    print(p)
    lf_tx = pl.scan_parquet(tx_path)
display(lf_tx.collect_schema())
dc = DataCatalog(
    src_tabts_path=tx_path,
    cat_path=tx_path.parent / "cat",
    meta_path=tx_path.parent / "meta",
    targets_path=tx_path.parent / "flag.parquet",
)
dc.print_setnames()
setname = "test"
groupby = "app_id"
display(pl.scan_parquet(dc.cat(setname=setname)).collect_schema())
cat_columns = list(read_json(dc.catnum_json(setname=setname)).keys())
print(cat_columns)
print(len(cat_columns))
num_columns = ["amnt", "hour_diff"]
print(num_columns)

True
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/encoded
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/__MACOSX
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/checkpoints
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/flag.parquet
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/cat
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/tb_logs
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/meta
/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/train_transactions_contest


Schema([('app_id', Int32),
        ('amnt', Float64),
        ('currency', Int32),
        ('operation_kind', Int32),
        ('card_type', Int32),
        ('operation_type', Int32),
        ('operation_type_group', Int32),
        ('ecommerce_flag', Int32),
        ('payment_system', Int32),
        ('income_flag', Int32),
        ('mcc', Int32),
        ('country', Int32),
        ('city', Int32),
        ('mcc_category', Int32),
        ('day_of_week', Int32),
        ('hour', Int32),
        ('days_before', Int32),
        ('weekofyear', Int32),
        ('hour_diff', Int64),
        ('transaction_number', Int32),
        ('__index_level_0__', Int64)])

allcat
test


Schema([('app_id', Int32),
        ('mcc', Int32),
        ('country', Int32),
        ('city', Int32),
        ('mcc_category', Int32)])

['mcc', 'country', 'city', 'mcc_category']
4
['amnt', 'hour_diff']


In [4]:
num_grous = read_num_grous(dc.targets, groupby="app_id")
num_chunk_per_worker = 10
chunk_reader = DDChunkReader(
    groupby="app_id",
    num_chunk_per_worker=num_chunk_per_worker,
    num_groups=num_grous,
    memlimit_gb=1.5,
)
lazy_dataset = LazyDataset(chunk_reader=chunk_reader)
lazy_dataset.set_catdata(
    data_path=dc.cat(setname),
    columns_to_select=cat_columns,
    catnum_json=dc.catnum_json(setname=setname),
)
lazy_dataset.set_numdata(data_path=dc.src, columns_to_select=num_columns)
lazy_dataset.set_targetmeta(data_path=dc.targets, target_column="flag")

In [5]:
train_ds, val_ds = random_split(lazy_dataset, test_size=0.05)
flag_counts = (
    pl.scan_parquet(dc.targets).select(pl.col("flag").unique_counts()).collect()
)
pos_weight = (flag_counts[1] / flag_counts[0]).item()
print(f"pos_weight={pos_weight}")
embedding_dims = lazy_dataset.embedding_dims_template(embedding_dim=20)
display(embedding_dims)
embedding_params = cat_embedding_params(
    catnum=lazy_dataset.catnum,
    col_embedding_dims=embedding_dims,
)

display(embedding_params)

pos_weight=0.02835684578237665


{'mcc': 20, 'country': 20, 'city': 20, 'mcc_category': 20}

{'mcc': CatColEmbeddingParams(num_embeddings=109, embedding_dim=20),
 'country': CatColEmbeddingParams(num_embeddings=25, embedding_dim=20),
 'city': CatColEmbeddingParams(num_embeddings=164, embedding_dim=20),
 'mcc_category': CatColEmbeddingParams(num_embeddings=29, embedding_dim=20)}

In [6]:
len(train_ds)

915729

In [ ]:
b_lstm = LitImbBinTabLSTM(
    cat_enc_params=embedding_params,
    h_size=30,
    # pos_weight=pos_weight,
    columns_by_type=lazy_dataset.type_columns,
    attention=True,
    pooling_type=PoolingType.LAST,
    #bidirectional=True,
    log_pr_with=LogPRMethod.add_figure,
    lr=1e-3,
    num_layers=2,
)

In [8]:
train_dl = DataLoader(
    dataset=train_ds,
    batch_size=50,
    collate_fn=collate_padded_batch_fn,
    num_workers=0,
)
val_dl = DataLoader(
    dataset=val_ds,
    batch_size=50,
    collate_fn=collate_padded_batch_fn,
    num_workers=0,
)

In [9]:
%%time
warnings.filterwarnings("ignore")
logger = TensorBoardLogger(save_dir=tx_path.parent / "tb_logs", name="att")
# profiler = PyTorchProfiler(
#     dirpath=tx_path.parent / "lit",
#     filename="profile_report.json",
#     export_to_chrome=True,
# )
display(logger.save_dir)
trainer = L.Trainer(
    limit_val_batches=300,
    log_every_n_steps=150,
    val_check_interval=200,
    max_epochs=50,
    accelerator="auto",
    devices=1,
    logger=logger,
    enable_progress_bar=True,
)
trainer.fit(model=b_lstm, train_dataloaders=train_dl, val_dataloaders=val_dl)

'/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/tb_logs'

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

CPU times: user 9 s, sys: 1.72 s, total: 10.7 s
Wall time: 3.38 s


RuntimeError: mat1 and mat2 shapes cannot be multiplied (101600x60 and 30x1)